# Alpha-Mind Quick Start 1: Introduction to Factor Processing

The alpha-mind data package provides functions for factor neutralization, standardization, and winsorization:
* [neutralize.py](https://github.com/alpha-miner/alpha-mind/blob/master/alphamind/data/neutralize.py): provides the risk-neutralization function *neutralize*.
* [standardize.py](https://github.com/alpha-miner/alpha-mind/blob/master/alphamind/data/standardize.py): provides *standardize*, the sklearn-style *Standardizer*, and *GroupStandardizer*.
* [winsorize.py](https://github.com/alpha-miner/alpha-mind/blob/master/alphamind/data/winsorize.py): provides the *winsorize_normal* outlier-control function.

Standardization and winsorization can also process data by group, for example by industry.

### 1. Winsorization
#### Full-Cross-Section Processing

The *winsorize_normal* function provided by alpha-mind assumes normally distributed data and controls extreme values.

In [ ]:
# Assume 50 stocks with one factor each, forming a matrix
# For demonstration, use one standard deviation as the lower and upper bounds
import asyncio
import datetime as dt


#### Industry-by-Industry Processing
Winsorization sometimes needs to be grouped, for example by industry. Pass an additional grouping argument to *winsorize_normal* for this purpose.

In [ ]:
# Assume two groups (such as industries), with the first and last 25 stocks in different groups
# Give the first and last 25 factors different standard deviations
# For demonstration, use one standard deviation as the lower and upper bounds
import numpy as np
import pyarrow as pa


### 2. Standardization
Standardization is used similarly to winsorization, so this section demonstrates only grouped standardization.

In [ ]:
# Assume two groups (such as industries), with the first and last 25 stocks in different groups
# Give the first and last 25 factors different standard deviations
# For demonstration, use one standard deviation as the lower and upper bounds
import pyarrow.compute as pc

rng = np.random.default_rng(42)


Sklearn-style *Standardizer* and *GroupedStandardizer* operators are also available. Their *fit* and *transform* methods produce the same effect. See the sklearn documentation and this project's source code for details.

### 3. Neutralization
#### Risk (Style) Neutralization

In [ ]:
# Assume 30 stocks with ten factors each, forming a matrix
# Assume four risk factors per stock, forming a matrix
# Risk neutralization is the residual from linearly regressing the original factor on risk factors
n_dates = 6
n_assets = 12


#### Risk and Industry Neutralization
Production environments generally also require industry neutralization. For *neutralize*, pass an additional NumPy array containing each stock's industry label.

In [ ]:
# Assume all stocks belong to three groups (such as industries)
# Neutralize the factor against risk and industry
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
# If every industry is identical, the result will equal ret_neutralize_1
# print(ret_neutralize_1)
# print(ret_neutralize_3)
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


#### Industry-Only Neutralization
For industry-only neutralization, pass a matrix of ones as the risk-factor argument, which is the first argument to *neutralize*. The following code provides a simple test.

In [ ]:
# Assume ten stocks with one factor each
# Assume two industries, with the first and last five stocks in different industries
from alphamind.data.processing import factor_processing
from alphamind.data.standardize import standardize


The industry-neutralized factor equals the original factor minus its industry mean, as expected.

### 4. Combined Preprocessing Function
Alpha-mind also provides *factor_processing*, which combines all the functions described above into a single preprocessing step.
- Pass a list of preprocessing functions to *pre_process*, such as winsorization and standardization. When risk factors are provided, neutralization runs automatically after preprocessing.
- *post_process* specifies processing after neutralization, if any, or after *pre_process*. Winsorization and standardization functions can again be supplied as a list.

In [ ]:
# Assume 30 stocks with ten factors each, forming a matrix
# Assume four risk factors per stock, forming a matrix
# Assume all stocks belong to three groups (such as industries)
from alphamind.data.winsorize import winsorize_normal

values = np.column_stack([
    panel["factor"].to_numpy(zero_copy_only=False),
    panel["value"].to_numpy(zero_copy_only=False),
])


In [ ]:
#ret_preprocess
risk = panel["SIZE"].to_numpy(zero_copy_only=False).reshape(-1, 1)
processed = factor_processing(
    values,
    pre_process=[winsorize_normal, standardize],
    risk_factors=risk,
    post_process=[standardize],
)


In [ ]:
assert np.isfinite(processed).all()
pa.table({"momentum": processed[:, 0], "value": processed[:, 1]})
